In [64]:
import pandas as pd
import numpy as np
df = pd.read_csv("indian_startups_fundingDADS.csv")
df.head()
df.columns
df.shape
df.info()
df.isnull().sum()
df.duplicated().sum()
df.columns = [
    "startup_name",
    "city",
    "industry",
    "funding_stage",
    "amount",
    "funding_date",
    "investors",
    "founded_year"
]
df.columns
df["startup_name"] = df["startup_name"].str.strip()
df["city"] = df["city"].str.strip()
df["startup_name"] = df["startup_name"].str.title()
df["city"] = df["city"].str.title()

city_mapping = {
    "Bangalore": "Bengaluru",
    "Blr": "Bengaluru",
    "Bombay": "Mumbai",
    "Gurgaon": "Gurugram",
    "Delhi": "New Delhi",
    "Calcutta": "Kolkata",
    "Madras": "Chennai"
}
df["city"] = df["city"].replace(city_mapping)
sorted(df["city"].unique())

industry_mapping = {
    "Fin Tech": "Fintech",
    "Fintech": "Fintech",
    "Ed Tech": "Edtech",
    "Health Tech": "HealthTech",
    "E-Commerce": "Ecommerce",
    "Food Tech": "FoodTech"
}
df["industry"] = df["industry"].replace(industry_mapping)
df["industry"].value_counts()
stage_mapping = {
    "Series-A": "Series A",
    "Series A Round": "Series A",
    "Seed Round": "Seed",
    "Pre-Series A": "Pre-Series A"
}
df["funding_stage"] = df["funding_stage"].replace(stage_mapping)
df["amount"] = (
    df["amount"]
    .str.replace("₹", "", regex=False)
    .str.replace("INR", "", regex=False)
    .str.replace("Cr", "", regex=False)
    .str.replace("Crore", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["amount"].head()
df["amount"].dtype
df["funding_date"] = pd.to_datetime(
    df["funding_date"],
    errors="coerce",
    dayfirst=True
)
df["funding_date"].head()
df["funding_date"].dt.year

df = df.drop_duplicates()
df.duplicated().sum()

df.info()
df.head()
df.columns
df.isnull().sum()

print(df.shape)
print(df.isnull().sum())

print(df.duplicated().sum())

print(sorted(df.columns))
df.columns

stats = {
    "count": df["founded_year"].count(),
    "mean": df["founded_year"].mean(),
    "min": df["founded_year"].min(),
    "max": df["founded_year"].max(),
    "median": df["founded_year"].median(),
    "std": df["founded_year"].std()
}

print(stats)

print(sorted(df["city"].unique()))

print(df["industry"].value_counts())

print(df["amount"].dtype)
print(df["amount"].isna().sum())
print(df["amount"].mean())
print(df["amount"].min())
print(df["amount"].max())

print(df["funding_date"].dt.year.value_counts())
print(df["funding_date"].dt.year.value_counts().idxmax())

print(df["funding_stage"].value_counts())

df["city"].unique()
df["city"].value_counts()
df["city"].isnull().sum()
df["city"].dtype
df["city"] == "Bengaluru"
result = df[df["amount"] > 100]

result = result.sort_values(by="amount", ascending=False)

print(result[["startup_name", "amount"]])
result = df[
    (df["city"] == "Bengaluru") &
    (df["amount"] > 100)
]

result = result.sort_values(by="amount", ascending=False)

print(result[["startup_name", "city", "amount"]])

result = df[df["industry"].isin(["Fintech", "Edtech", "HealthTech"])]

print(result.shape[0])

result = df[df["founded_year"].between(2015, 2020)]

print(result.shape[0])

result = df[df["startup_name"].str.contains("pay", case=False, na=False)]

print(result["startup_name"].unique())

df["funding_efficiency"] = df["amount"] / (2024 - df["founded_year"])

print(
    df[
        [
            "startup_name",
            "industry",
            "amount",
            "founded_year",
            "funding_efficiency"
        ]
    ].sort_values(
        by="funding_efficiency",
        ascending=False
    ).head(10)
)

df["funding_tier"] = df["amount"].apply(
    lambda x:
        "Unicorn" if x >= 1000 else
        "Mega" if x >= 500 else
        "Large" if x >= 100 else
        "Mid" if x >= 10 else
        "Small / Unknown"
)

print(df["funding_tier"].value_counts())

df["era"] = pd.cut(
    df["founded_year"],
    bins=[0, 2009, 2017, 2025],
    labels=["Early", "Growth", "Recent"]
)

print(df["era"].value_counts())

df["investor_count"] = df["investors"].apply(
    lambda x: len(x.split(", ")) if pd.notna(x) else 0
)

print(df["investor_count"].value_counts())

avg_funding = (
    df.groupby("industry")["amount"]
      .mean()
      .sort_values(ascending=False)
)

print(avg_funding.head(5))

city_summary = (
    df.groupby("city")
      .agg({
          "startup_name": "count",
          "amount": ["sum", "mean"]
      })
)

print(city_summary)

industry_stats = (
    df.groupby("industry")
      .agg(
          startup_count=("startup_name", "count"),
          avg_funding=("amount", "mean")
      )
)
industry_stats = industry_stats[
    industry_stats["startup_count"] >= 15
]
print(
    industry_stats.sort_values(
        by="avg_funding",
        ascending=False
    ).head(1)
)
top5 = (
    df["industry"]
    .value_counts()
    .head(5)
    .index
)

result = pd.crosstab(
    df[df["industry"].isin(top5)]["industry"],
    df[df["industry"].isin(top5)]["funding_tier"]
)

print(result)

index = df.groupby("industry")["amount"].idxmax()

result = df.loc[
    index,
    ["startup_name", "industry", "city", "amount"]
]

print(result)
df.loc[index]

result = df[
    (df["founded_year"] >= 2019)
    &
    (df["amount"] >= 100)
    &
    (
        df["industry"].isin(
            ["Fintech", "Edtech", "HealthTech"]
        )
    )
]

print(
    result[
        [
            "startup_name",
            "industry",
            "city",
            "founded_year",
            "amount"
        ]
    ].sort_values(
        by="amount",
        ascending=False
    )
)

city = (
    df.groupby("city")
      .agg(
          startup_count=("startup_name", "count"),
          total_funding=("amount", "sum")
      )
)

city = city[
    city["startup_count"] >= 20
]

print(
    city.sort_values(
        by="total_funding",
        ascending=False
    ).head(1)
)
result = (
    df.groupby(
        ["city", "industry"]
    )
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print(result)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 468 entries, 0 to 467
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Startup Name   468 non-null    object 
 1   City           468 non-null    object 
 2   INDUSTRY       468 non-null    object 
 3   Funding Stage  468 non-null    object 
 4   Amount         468 non-null    object 
 5   Funding Date   449 non-null    object 
 6   Investors      458 non-null    object 
 7   Founded Year   445 non-null    float64
dtypes: float64(1), object(7)
memory usage: 29.4+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 450 entries, 0 to 467
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   startup_name   450 non-null    object        
 1   city           450 non-null    object        
 2   industry       450 non-null    object        
 3   funding_stage  450 non-null    object       